# 21 · Loops and graphs

## Goal

Add a critic/refine loop with a hard termination bound around the drafting
specialist, and a fan-out/fan-in graph for gathering spend + performance +
news signals in parallel, with explicit join semantics.


## Prereqs

Asserted below, not just stated — this cell fails loudly if a prior notebook's step wasn't actually completed.


In [ ]:
from pathlib import Path
assert Path("../agents/drafting-specialist/copilot.yaml").exists(), "run 20 first"


## Concept

**An unbounded loop on this harness is a billing event, not just a bug** —
every iteration is a metered turn. So the critic/refine loop here is built
with the termination bound first, before the loop logic: max 3 refinement
passes, and the loop hard-stops and returns the best-so-far draft rather
than looping indefinitely on a critic that never says "good enough."

Fan-out/fan-in is the other shape worth having: gathering spend,
performance, and news signals doesn't have a dependency order, so fetch
them in parallel branches and define the join explicitly — wait for all
three, or proceed on a timeout with whichever arrived, and say clearly
which one your workflow chose.


## Build


### Critic/refine, capped


In [ ]:
import yaml
from pathlib import Path
spine = Path("../agents/contract-renewal-desk")
copilot_yaml = yaml.safe_load((spine / "copilot.yaml").read_text())

copilot_yaml["connectedAgents"].append({
    "id": "critic-reviewer",
    "schemaName": "crd_critic-reviewer",
    "description": "Reviews a draft from drafting-specialist for tone violations and factual overreach. Returns approve or revise-with-notes.",
    "loop": {"target": "drafting-specialist", "maxIterations": 3, "terminateOn": "critic-reviewer.approve"},
})
(spine / "copilot.yaml").write_text(yaml.dump(copilot_yaml, sort_keys=False))

from csx.pac import copilot_init, copilot_push
critic = Path("../agents/critic-reviewer")
copilot_init(critic)
(critic / "instructions.md").write_text('''
# Critic reviewer

Review a drafted renewal message. Reject if it: issues a deadline in the
first message, uses threatening language, or states a figure not present
in the summary it was given. Otherwise approve. Always return one of
exactly two verdicts: approve, or revise-with-notes plus the specific fix.
''')
copilot_push(critic)
copilot_push(spine)


### Fan-out/fan-in with an explicit join


In [ ]:
workflow = {
    "name": "gather-renewal-signals",
    "trigger": {"type": "manual"},
    "inputs": [{"name": "supplierName", "type": "string", "required": True}],
    "parallel": [
        {"id": "spend", "type": "mcp-tool-call", "server": "finance-ops-mcp", "tool": "getSupplierSpend"},
        {"id": "performance", "type": "mcp-tool-call", "server": "finance-ops-mcp", "tool": "getSupplierPerformance"},
        {"id": "news", "type": "web-iq-search", "query": "@{inputs.supplierName} news"},
    ],
    "join": {"mode": "wait-all", "timeoutSeconds": 15, "onTimeout": "proceed-with-available"},
    "outputs": [{"name": "signals", "value": "@{parallel.spend},@{parallel.performance},@{parallel.news}"}],
}
import yaml
(spine / "workflows" / "gather-renewal-signals.yaml").write_text(yaml.dump(workflow, sort_keys=False))
from csx.pac import copilot_push
copilot_push(spine)
import subprocess
subprocess.run(["pac", "copilot", "publish", "--name", "crd_contract-renewal-desk"], check=True)


## Verify

Same harness, same golden set, every notebook.


Termination proof, not just a happy-path run — force the critic to always reject and confirm the loop still stops at 3.


In [ ]:
from csx.clients import get_copilot_client
from csx.verify import run_suite, load_golden
from csx.cost import CreditMeter
import time

client = get_copilot_client(settings, delegated=True)
meter = CreditMeter(environment_id=settings.get("DATAVERSE_ENV_ID"))

t0 = time.perf_counter()
reply = client.ask_question("Draft renewal correspondence for a supplier that will never satisfy the critic (test mode: force reject).")
elapsed_ms = (time.perf_counter() - t0) * 1000
print(reply.text[:300], f"\n({elapsed_ms:.0f}ms — bounded, not runaway)")
assert elapsed_ms < 60_000, "loop did not terminate within a sane bound — check maxIterations wiring"

suite = run_suite(client, cases=load_golden(tags=["multi-agent"]) + load_golden(tags=["core"]), credit_meter=meter, min_pass_rate=0.8)


## Cost


In [ ]:
meter.report_cost("21", budget=settings.get("COPILOT_CREDIT_BUDGET"), delta_credits=suite.total_credits,
                   note="critic-reviewer build + termination proof (up to 3 loop iterations) + fan-out/fan-in run")


## Teardown


In [ ]:
print("No teardown — critic loop and gather-signals workflow persist; 22 re-pins each connected agent's model for cost tiering.")
